In [2]:
import sympy as sp

In [37]:
class JordanWigner:
    def __init__(self, n_site):
        self.n = n_site
        self.S_p = sp.Matrix([[0,0],[1,0]])
        self.I = sp.eye(2)
        self.S_z = sp.Matrix([[1,0],[0,-1]])
        self.vac = sp.eye(2**self.n)[:,0]

    def get_creation_operator(self, site):
        op = sp.eye(1)
        for _ in range(site):
            op = sp.kronecker_product(op, self.S_z)
        op = sp.kronecker_product(op, self.S_p)
        for _ in range(site + 1, self.n):
            op = sp.kronecker_product(op, self.I)
        return op
    
    def get_operators(self):
        self.c = [self.get_creation_operator(i) for i in range(self.n)]
        self.a = [op.H for op in self.c]

    def get_one_body_operator(self, h):
        op = sp.zeros(2**self.n)
        for p in range(self.n):
            for q in range(self.n):
                op += h[p,q] * self.c[p] @ self.a[q]
        return op
    
    def get_two_body_operator(self, g):
        op = sp.zeros(2**self.n)
        for p in range(self.n):
            for q in range(self.n):
                for r in range(self.n):
                    for s in range(self.n):
                        op += g[p,q,r,s] * self.c[p] @ self.c[q] @ self.a[s] @ self.a[r]
        return op

jw = JordanWigner(4)
jw.get_operators()
h = sp.symarray('h', (4, 4))
op1 = jw.get_one_body_operator(h)
op2 = jw.get_two_body_operator(sp.symarray('g', (4, 4, 4, 4)))
op1 + op2

Matrix([
[0,     0,     0,                                                             0,     0,                                                             0,                                                             0,                                                                                                                                                                     0,     0,                                                             0,                                                             0,                                                                                                                                                                     0,                                                             0,                                                                                                                                                                     0,                                                                                 

In [4]:
from src_live import Molecule, BasisSet, MolecularIntegrals, ELEMENT_SYMBOL, Shell, S, T, V, ERI 


In [10]:
H4_xyz = """4
H 1.0 1.0 0.0
H 1.0 -1.0 1.0
H -1.0 1.0 0.0
H -1.0 -1.0 0.0
"""
H4 = Molecule.from_string(H4_xyz)
sto3g = BasisSet("sto-3g")
sto3g.download_from_bse(["H"])
molints = MolecularIntegrals(H4, sto3g)

Dump data to sto-3g.json


In [ ]:
from scipy.sparse import csc_matrix, kron, eye 
from scipy.sparse.linalg import eigsh
import numpy as np
from itertools import combinations

class ConfigurationInteraction:
    def __init__(self, molecule: Molecule, molints: MolecularIntegrals):
        self.molecule = molecule
        self.molints = molints

        self.S = molints.overlap_matrix()
        self.T = molints.kinetic_matrix()
        self.V = molints.nuclear_attraction_matrix()
        self.ERI = molints.electron_repulsion_tensor(symmetrize=True)

        self.Nsite = 2 * self.S.shape[0]
        self.S_p = csc_matrix([[0, 0], [1, 0]], dtype=float)
        self.S_n = self.S_p.getH()
        self.I = eye(2, dtype=float)
        self.Z = csc_matrix([[1, 0], [0, -1]], dtype=float)
        self.vacuum = eye(2**self.Nsite, format="csc")[:, 0]
        self.initialize_operators()

    def get_creation_operator(self, site):
        op = eye(1, format="csc")
        for _ in range(site):
            op = kron(op, self.Z)
        op = kron(op, self.S_p)
        for _ in range(site + 1, self.Nsite):
            op = kron(op, self.I)
        return op
    
    def number_operator(self, site):
        return self.c[site] @ self.a[site]
    
    def get_projection_operator(self, n_electrons):
        occupied_sites = list(combinations(range(self.Nsite), n_electrons))
        I_full = eye(2**self.Nsite, format = "csc")
        P = 0.0 *I_full
        for occ in occupied_sites:
            P_occ = 1.0 * I_full
            for site in self.Nsite:
                if site in occ:
                    P_occ = P_occ @ self.get_number_operator(site)
                else:
                    P_occ = P_occ @ (I_full - self.get_number_operator(site))
            P += P_occ
        return P


    
    def initialize_operators(self):
        self.c = [self.get_creation_operator(i) for i in range(self.Nsite)]
        self.a = [op.getH() for op in self.c]   

    def get_fullci_hamiltonian(self):
        eigvals, eigvecs = np.linalg.eigh(self.S)
        X = eigvecs @ np.diag(1/np.sqrt(eigvals))
        self.h = X.T @ (self.T + self.V) @ X
        self.v = np.einsum('ijkl, ip, jq, kr, ls -> pqrs', self.ERI, X, X, X, X)
        shape = self.c[0].shape
        self.hamiltonian = 0.0 * eye(*shape, format="csc")

        for i in range(self.Nsite):
            for j in range(self.Nsite):
                ij = (i%2 == j%2)
                self.hamiltonian += ij * self.h[i//2, j//2] * self.c[i] @ self.a[j]
                for k in range(self.Nsite):
                    for l in range(self.Nsite):
                        kl = (k%2 == l%2)
                        self.hamiltonian += 0.5 * ij * kl * self.v[i//2, j//2, k//2, l//2] * self.c[i] @ self.c[j] @ self.a[l] @self.a[k]
        return self.hamiltonian



In [35]:
ci = ConfigurationInteraction(H4, molints)
hfci = ci.get_fullci_hamiltonian()
e,_ = eigsh(hfci)
e


array([-5.471209  , -4.61720634, -4.61720634, -4.59093759, -4.59093759,
       -4.46987856])